In [ ]:
!pip install cirq --quiet
!pip install cirq-web --quiet
!pip install openfermion
!pip install openfermionpyscf
!pip install mealpy

import numpy as np
import matplotlib.pyplot as plt
import sympy
import openfermion as of
import openfermionpyscf as ofpyscf
import mealpy
import math

from scipy.optimize import minimize
from scipy.linalg import eigh

def binary_labels(num_qubits):
    return [bin(x)[2:].zfill(num_qubits) for x in range(2 ** num_qubits)]
plt.rcParams.update({'font.size': 8})

import cirq
from cirq_web import bloch_sphere
from cirq import Z, PauliSum
print("installed cirq.")


import warnings
warnings.filterwarnings("ignore")

ModuleNotFoundError: No module named 'numpy'

In [ ]:
#setting up the molecule's attributes
geometry = [['H', [0.0, 0.0, 0.0]], ['H', [0.0, 0.0, 0.74]]]
basis = 'sto-3g'
multiplicity = 1
charge = 0

#utilizing openfermion and piscyf to create the hamiltonians
hamiltonian = ofpyscf.generate_molecular_hamiltonian(geometry, basis, multiplicity, charge)
hamiltonian_qubit = of.jordan_wigner(of.get_fermion_operator(hamiltonian))
hamiltonian_matrix = of.get_sparse_operator(hamiltonian_qubit)

In [ ]:
#setting up ansatz with 4 qubits
def ansatz(angle):
  qubits = cirq.NamedQubit.range(4, prefix='q')
  circuit = cirq.Circuit()
  circuit.append([cirq.rx(math.radians(angle[0])).on(qubits[0]), cirq.rx(math.radians(angle[0])).on(qubits[1]), cirq.rx(math.radians(angle[0])).on(qubits[2]), cirq.rx(math.radians(angle[0])).on(qubits[3])])
  circuit.append([cirq.CNOT(qubits[0], qubits[1]), cirq.CNOT(qubits[2], qubits[3])])
  return circuit, qubits

In [ ]:
sim = cirq.Simulator()
#setting up cost function
def expectation_value(angle):
  #getting a state vector from ansatz
  circuit, qubits = ansatz(angle)
  result = sim.simulate(circuit)
  vector = result.final_state_vector

  #finding the total energy of the molecule through matrix multiplication
  energy = np.dot(vector.conj(), hamiltonian_matrix.dot(vector))

  #returning only the real value of the energy
  return energy.real

In [ ]:
#defining the problem by setting angle bounds and setting the energy function as the objective
problem_dict = {"bounds": mealpy.FloatVar(lb=-math.pi, ub=math.pi, name="delta"), "obj_func": expectation_value, "minmax": "min",}

#setting up the Emperor Pengiun Optimizer
optimizer = mealpy.EPC.DevEPC(epoch = 1000, pop_size = 50)

#utilizing the optimizer to find the best solution
best = optimizer.solve(problem_dict)

print("Best angle:", best.solution[0])
print("Lowest energy:", best.target.fitness)